# Scores for the penalty-based two-courier QAOA notebook

This notebook computes the leaderboard measures from `solutions/README.md` for `Eli_two_couriers_solution.ipynb` in the repository root.

That notebook solves a smaller and different instance than the golden notebook:

- **Locations:** the depot and stops 1 to 3.
- **Distances:** the real one-way distances.
- **Rule for the second van:** both couriers must deliver at least one stop. There is no distance limit.
- **Encoding:** one qubit per "courier c visits stop d at position p", 18 qubits. Invalid bitstrings are made expensive with a penalty of 100.

The circuit is re-simulated here from the angles printed in that notebook. The cost only contains Z terms, so every bitstring simply has an energy, and one layer is a phase per bitstring followed by an X rotation on every qubit.

In [1]:
import itertools
import json
import math
from datetime import datetime
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

STOPS = [1, 2, 3]
COURIERS = 2
PENALTY = 100.0
SCALE = 100.0
M = len(STOPS)
N_QUBITS = COURIERS * M * M

DATA_FILE = next(path for path in (Path(up) / "tallinn_osrm_matrix.json" for up in (".", "..", "../..")) if path.exists())
ROAD_KM = np.array(json.loads(DATA_FILE.read_text(encoding="utf-8"))["distances"], dtype=float)[:M + 1, :M + 1] / 1000

# Qubit of "courier c visits stop d at position p", in the order the notebook creates its variables.
QUBIT = {(c, p, d): c * M * M + p * M + k for c in range(COURIERS) for p in range(M) for k, d in enumerate(STOPS)}


def build_qubo():
    """The QUBO matrix of the notebook: four penalty rules, then the route distances."""
    q = np.zeros((N_QUBITS, N_QUBITS))

    def squared(qubits, weight):
        """weight * (sum of the qubits - 1)^2, without its constant."""
        for i in qubits:
            q[i, i] -= weight
        for i, j in itertools.combinations(qubits, 2):
            q[i, j] += 2 * weight

    for d in STOPS:                                    # every stop is visited once
        squared([QUBIT[c, p, d] for c in range(COURIERS) for p in range(M)], PENALTY)
    for c in range(COURIERS):
        for p in range(M):                             # at most one stop per position
            for a, b in itertools.combinations(STOPS, 2):
                q[QUBIT[c, p, a], QUBIT[c, p, b]] += PENALTY
        for p in range(M - 1):                         # no empty position before a used one
            for after in STOPS:
                q[QUBIT[c, p + 1, after], QUBIT[c, p + 1, after]] += PENALTY
                for before in STOPS:
                    q[QUBIT[c, p, before], QUBIT[c, p + 1, after]] -= PENALTY
        squared([QUBIT[c, 0, d] for d in STOPS], PENALTY)   # each courier has a first stop

    for c in range(COURIERS):
        for d in STOPS:
            q[QUBIT[c, 0, d], QUBIT[c, 0, d]] += ROAD_KM[0, d]
        for p in range(M - 1):
            for a in STOPS:
                for b in STOPS:
                    q[QUBIT[c, p, a], QUBIT[c, p + 1, b]] += ROAD_KM[a, b]
        for p in range(M):                             # the way back to the depot counts for the last stop only
            for d in STOPS:
                q[QUBIT[c, p, d], QUBIT[c, p, d]] += ROAD_KM[d, 0]
                if p < M - 1:
                    for after in STOPS:
                        q[QUBIT[c, p, d], QUBIT[c, p + 1, after]] -= ROAD_KM[d, 0]
    return q


def route_km(route):
    path = [0, *route, 0]
    return sum(ROAD_KM[a, b] for a, b in zip(path, path[1:]))


Q = build_qubo()
bits = ((np.arange(2 ** N_QUBITS)[:, None] >> np.arange(N_QUBITS)) & 1).astype(float)
qubo_value = np.einsum("bi,ij,bj->b", bits, Q, bits)
ising_constant = np.trace(Q) / 2 + np.triu(Q, 1).sum() / 4
ENERGY = (qubo_value - ising_constant) / SCALE       # the scaled Hamiltonian of the notebook

# Every valid plan: both couriers have at least one stop, and every stop is visited once.
plans = {}
for first in (group for size in range(1, M) for group in itertools.combinations(STOPS, size)):
    second = [d for d in STOPS if d not in first]
    for route_1 in itertools.permutations(first):
        for route_2 in itertools.permutations(second):
            state = sum(1 << QUBIT[c, p, d] for c, route in enumerate((route_1, route_2)) for p, d in enumerate(route))
            plans[state] = (route_km(route_1) + route_km(route_2), route_1, route_2)
VALID = np.array(sorted(plans))
VALID_KM = np.array([plans[state][0] for state in VALID])
SHORTEST = VALID_KM.min()
OPTIMAL = VALID[VALID_KM <= SHORTEST + 1e-9]

offset = PENALTY * (M + COURIERS)
print(f"{len(VALID)} valid plans among {2 ** N_QUBITS:,} bitstrings, {len(OPTIMAL)} shortest: {SHORTEST:.3f} km, "
      f"routes {plans[OPTIMAL[0]][1]} and {plans[OPTIMAL[0]][2]}")
print(f"The QUBO equals the total distance on every valid plan: {np.allclose(qubo_value[VALID] + offset, VALID_KM)}")
print(f"Invalid bitstrings with a lower cost than the shortest plan: {int((qubo_value < qubo_value[VALID].min() - 1e-9).sum())}")


def qaoa_probabilities(gamma, beta):
    """Probability of every bitstring after one layer of standard QAOA (X mixer)."""
    state = (np.exp(-1j * gamma * ENERGY) / math.sqrt(2 ** N_QUBITS)).reshape((2,) * N_QUBITS)
    for axis in range(N_QUBITS):
        state = math.cos(beta) * state - 1j * math.sin(beta) * np.flip(state, axis=axis)
    return np.abs(state.reshape(-1)) ** 2


def shots_for_99(p_opt):
    return math.inf if p_opt <= 0 else math.ceil(math.log(0.01) / math.log1p(-p_opt))


def measures(probabilities, layers):
    p_valid, p_opt = probabilities[VALID].sum(), probabilities[OPTIMAL].sum()
    shots = shots_for_99(p_opt)
    return {"P(valid)": p_valid, "P(opt)": p_opt, "mean gap": probabilities[VALID] @ VALID_KM / p_valid / SHORTEST - 1,
            "shots": shots, "evaluations": shots * (layers + 1)}

12 valid plans among 262,144 bitstrings, 2 shortest: 15.631 km, routes (1, 3) and (2,)
The QUBO equals the total distance on every valid plan: True
Invalid bitstrings with a lower cost than the shortest plan: 0


# Eli's notebook: 1 layer

Angles from the output of the "Optimize QAOA Parameters" cell at commit `2641ac8`. That notebook reports a final energy of −1.947230, 0.000061 % for its most likely valid plan, and no valid plan in 10,000 shots. This simulation must reproduce the first two.

In [2]:
GAMMA, BETA = 0.794934, 0.330698
eli_probabilities = qaoa_probabilities(GAMMA, BETA)
eli = measures(eli_probabilities, layers=1)
print(f"Energy = {eli_probabilities @ ENERGY:.6f}   most likely valid plan = {eli_probabilities[VALID].max():.6%}")
print(f"P(valid) = {eli['P(valid)']:.5%}   P(opt) = {eli['P(opt)']:.5%}   mean gap = {eli['mean gap']:.1%}")
print(f"Expected valid shots in 10,000: {10_000 * eli['P(valid)']:.2f}")

Energy = -1.947230   most likely valid plan = 0.000061%
P(valid) = 0.00073%   P(opt) = 0.00012%   mean gap = 13.4%
Expected valid shots in 10,000: 0.07


# Why no valid shot appears

The model is right: the check above shows that the cost of every valid plan is its distance, and that no invalid bitstring is cheaper than the shortest plan. The circuit is the weak part. Three numbers show it:

- **Against random bitstrings.** The circuit returns a valid plan less often than a uniformly random 18-bit string.
- **The spread of the energies.** The penalties make the scaled energies span a range that is about 2,000 times wider than the differences between valid plans. The circuit reacts to the penalties and hardly sees the distances.
- **The best one-layer angles.** A grid over both angles, scored by P(valid), shows how far one layer can get. The notebook's optimiser minimised the average energy instead, with at most 30 evaluations. A lower average energy does not mean more valid shots.

In [3]:
uniform = measures(np.full(2 ** N_QUBITS, 2.0 ** -N_QUBITS), layers=0)
print(f"P(valid): circuit {eli['P(valid)']:.5%}, random bitstring {uniform['P(valid)']:.5%}, so the circuit is {uniform['P(valid)'] / eli['P(valid)']:.1f} times worse")
print(f"Scaled energies: from {ENERGY.min():.1f} to {ENERGY.max():.1f}, standard deviation {ENERGY.std():.1f}. "
      f"Valid plans differ by at most {(VALID_KM.max() - VALID_KM.min()) / SCALE:.3f}.")

grid = [(qaoa_probabilities(gamma, beta)[VALID].sum(), gamma, beta) for gamma in np.linspace(0.02, 1.0, 50) for beta in np.linspace(0.05, 1.55, 21)]
best_p_valid, best_gamma, best_beta = max(grid)
best = measures(qaoa_probabilities(best_gamma, best_beta), layers=1)
print(f"Best P(valid) on a {len(grid)}-point grid of one-layer angles: {best_p_valid:.5%} at gamma = {best_gamma:.2f}, beta = {best_beta:.2f} "
      f"({best_p_valid / uniform['P(valid)']:.1f} times a random bitstring), with P(opt) = {best['P(opt)']:.5%}")

P(valid): circuit 0.00073%, random bitstring 0.00458%, so the circuit is 6.3 times worse
Scaled energies: from -20.2 to 57.4, standard deviation 9.5. Valid plans differ by at most 0.035.


Best P(valid) on a 1050-point grid of one-layer angles: 0.05381% at gamma = 0.80, beta = 1.25 (11.8 times a random bitstring), with P(opt) = 0.00885%


# Leaderboard rows

Random guessing here means a uniformly random 18-bit string, which is what the circuit starts from. Brute force checks the 12 valid plans.

In [4]:
def pct(value):
    return f"{100 * value:.1f} %" if value >= 0.01 else f"{100 * value:.2g} %"


rows = [[name, layers, pct(m["P(opt)"]), pct(m["P(valid)"]), pct(m["mean gap"]), f"{m['shots']:,}", f"{m['evaluations']:,}"]
        for name, layers, m in (("Random 18-bit string", 0, uniform), ("Eli, simulated", 1, eli), ("Eli's circuit with the best grid angles", 1, best))]
headers = ["Solution", "Layers", "P(opt)", "P(valid)", "Mean gap", "Shots for 99 %", "Cost evaluations for 99 %"]
display(Markdown("\n".join(["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
                           + ["| " + " | ".join(str(cell) for cell in row) + " |" for row in rows])))

zz_terms = int(np.count_nonzero(np.triu(Q, 1)))
print(f"{zz_terms} ZZ terms per layer, so {2 * zz_terms} CX gates per layer before routing")

# The same four locations under the rule of the golden notebook: no van drives more than 75 % of the shortest one-van tour.
one_van = min(route_km(order) for order in itertools.permutations(STOPS))
limit = 0.75 * one_van
within = [(km, r1, r2) for km, r1, r2 in plans.values() if max(route_km(r1), route_km(r2)) <= limit]
print(f"With a distance limit instead ({limit:.3f} km, 75 % of the {one_van:.3f} km one-van tour): {len(within)} of the {len(plans)} plans stay valid, "
      f"and the shortest is {min(within)[0]:.3f} km, routes {min(within)[1]} and {min(within)[2]}")
print(f"Evaluated {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}")

| Solution | Layers | P(opt) | P(valid) | Mean gap | Shots for 99 % | Cost evaluations for 99 % |
|---|---|---|---|---|---|---|
| Random 18-bit string | 0 | 0.00076 % | 0.0046 % | 13.4 % | 603,607 | 603,607 |
| Eli, simulated | 1 | 0.00012 % | 0.00073 % | 13.4 % | 3,764,970 | 7,529,940 |
| Eli's circuit with the best grid angles | 1 | 0.0088 % | 0.054 % | 13.5 % | 52,043 | 104,086 |

87 ZZ terms per layer, so 174 CX gates per layer before routing
With a distance limit instead (10.976 km, 75 % of the 14.634 km one-van tour): 8 of the 12 plans stay valid, and the shortest is 15.631 km, routes (1, 3) and (2,)
Evaluated 2026-11-12 21:34 EET
